# Separate Model Execution Metrics Plotter

Generates **separate image files** for each training/execution metric across a list of `results/` folders and `(et, eta)` regions:
1. **Loss Curves** (`train_loss`, `val_loss`)
2. **Accuracy Curves** (`train_acc`, `val_acc`)
3. **Performance Curves** (`SP`, `PD`, `PF` with best-SP epoch marker)
4. **ROC Curve** (`RocPlotter`)
5. **Cross-Validation Fold Boxplots** (`BoxplotSPPlotter` for `SP`, `PD`, `PF`)

In [9]:
import gc
import re
import shutil
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from src.core.Plotting.Context import RegionPlotContext
from src.core.Plotting.Localization import PlotTranslator
from src.core.Plotting.MetricPlotter import (
    BoxplotSPPlotter,
    ModelMetricsPlotter,
    RocPlotter,
)
from src.core.Validation.ResultAggregator import ResultAggregator

print(f"Project root resolved: {PROJECT_ROOT}")

Project root resolved: /eos/home-i04/p/pmourafr/egamma_master


### 1. Configuration & Region Selection
Set `RESULTS_FOLDERS`, `TARGET_REGIONS` (list of `(et, eta)` pairs or `None` for all regions), `LANGUAGE`, and output directory.

In [10]:
RESULTS_FOLDERS = [
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_100Rings_id20261105222012",
    # PROJECT_ROOT / "results/config_nameModelV9_HighBatch_50Rings_id20261109122259",
    # PROJECT_ROOT / "results/config_nameModelV10_HighBatch_100Rings_id20261112210846",
    # PROJECT_ROOT / "results/config_nameModelV10_HighBatch_50Rings_id20261115051346",
]

# List of (et, eta) pairs to plot, or None to process all discovered regions
TARGET_REGIONS = [(0, 4)]

LANGUAGE = PlotTranslator.validate_language("pt")
INCLUDE_BOXPLOT = True
BOXPLOT_METRICS = ["best_sp_value", "best_pd_value", "best_fa_value"]

OUTPUT_DIR = PROJECT_ROOT / "Plots" / "SeparateModelMetrics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_plotter = ModelMetricsPlotter(language=LANGUAGE)
roc_plotter = RocPlotter(language=LANGUAGE)
boxplot_plotter = BoxplotSPPlotter(language=LANGUAGE)

for folder in RESULTS_FOLDERS:
    print(f"Results folder: {folder.name} (exists={folder.exists()})")

Results folder: config_nameModelV9_HighBatch_100Rings_id20261105222012 (exists=True)


### 2. Region Discovery & Streaming Helper
Helper functions to resolve `(et, eta)` regions in each results folder and build `RegionPlotContext` without caching all regions in RAM.

In [11]:
def resolve_regions(aggregator: ResultAggregator, target_regions) -> list[tuple[int, int]]:
    """Returns sorted (et, eta) pairs available in `aggregator`, filtered by `target_regions`."""
    available = []
    for key in aggregator._discover_region_files():
        match = re.match(r"^iet(\d+)\.ieta(\d+)$", key)
        if match:
            available.append((int(match.group(1)), int(match.group(2))))
    available.sort()
    if target_regions is None:
        return available
    target_set = {tuple(pair) for pair in target_regions}
    return [pair for pair in available if pair in target_set]

def make_context(iet: int, ieta: int, model_output_dir: Path) -> RegionPlotContext:
    return RegionPlotContext(
        iet=iet,
        ieta=ieta,
        output_dir=model_output_dir,
        data=np.array([]),
        target=np.array([]),
        language=LANGUAGE,
    )

def extract_model_tag(folder: Path) -> str:
    """Extracts a concise model tag from a results folder name."""
    name = folder.name
    if name.startswith("config_name"):
        name = name[len("config_name"):]
    return re.sub(r"_id\d+$", "", name)

def move_to_standard_name(src_path: str | Path, dest_dir: Path, filename: str) -> Path:
    """Moves a generated plotter output file to the standardized folder and filename."""
    src = Path(src_path)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / filename
    shutil.move(str(src), str(dest))
    return dest

### 3. Task 1: Loss Curves (`train_loss`, `val_loss`)

In [12]:
for folder in RESULTS_FOLDERS:
    if not folder.exists():
        print(f"Skipping missing folder: {folder}")
        continue
    agg = ResultAggregator(folder)
    model_tag = extract_model_tag(folder)
    model_out = OUTPUT_DIR / folder.name
    for iet, ieta in resolve_regions(agg, TARGET_REGIONS):
        try:
            best_model, _, _, _ = agg.get_best_model_for_region(iet, ieta, cache=False)
            saved = metrics_plotter.plot_separate(
                make_context(iet, ieta, model_out),
                history_data=best_model.get("history"),
                metrics=("loss",),
            )
            dest = move_to_standard_name(
                saved["loss"],
                model_out / "Loss",
                f"et_{iet}_eta_{ieta}_{model_tag}_loss_plot.pdf",
            )
            print(f"[{folder.name}] et_{iet}_eta_{ieta} loss -> {dest}")
        except Exception as exc:
            print(f"[{folder.name}] iet{iet}.ieta{ieta} skipped: {exc}")
        finally:
            gc.collect()

[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 loss -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/model_metrics/iet0.ieta4_loss.pdf


### 4. Task 2: Accuracy Curves (`train_acc`, `val_acc`)

In [13]:
for folder in RESULTS_FOLDERS:
    if not folder.exists():
        print(f"Skipping missing folder: {folder}")
        continue
    agg = ResultAggregator(folder)
    model_tag = extract_model_tag(folder)
    model_out = OUTPUT_DIR / folder.name
    for iet, ieta in resolve_regions(agg, TARGET_REGIONS):
        try:
            best_model, _, _, _ = agg.get_best_model_for_region(iet, ieta, cache=False)
            saved = metrics_plotter.plot_separate(
                make_context(iet, ieta, model_out),
                history_data=best_model.get("history"),
                metrics=("accuracy",),
            )
            dest = move_to_standard_name(
                saved["accuracy"],
                model_out / "Accuracy",
                f"et_{iet}_eta_{ieta}_{model_tag}_accuracy_plot.pdf",
            )
            print(f"[{folder.name}] et_{iet}_eta_{ieta} accuracy -> {dest}")
        except Exception as exc:
            print(f"[{folder.name}] iet{iet}.ieta{ieta} skipped: {exc}")
        finally:
            gc.collect()

[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 accuracy -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/model_metrics/iet0.ieta4_accuracy.pdf


### 5. Task 3: Operating Performance Curves (`SP`, `PD`, `PF`)

In [14]:
for folder in RESULTS_FOLDERS:
    if not folder.exists():
        print(f"Skipping missing folder: {folder}")
        continue
    agg = ResultAggregator(folder)
    model_tag = extract_model_tag(folder)
    model_out = OUTPUT_DIR / folder.name
    for iet, ieta in resolve_regions(agg, TARGET_REGIONS):
        try:
            best_model, _, _, _ = agg.get_best_model_for_region(iet, ieta, cache=False)
            saved = metrics_plotter.plot_separate(
                make_context(iet, ieta, model_out),
                history_data=best_model.get("history"),
                metrics=("performance",),
            )
            dest = move_to_standard_name(
                saved["performance"],
                model_out / "Performance",
                f"et_{iet}_eta_{ieta}_{model_tag}_performance_sp_pd_pf_plot.pdf",
            )
            print(f"[{folder.name}] et_{iet}_eta_{ieta} SP/PD/PF -> {dest}")
        except Exception as exc:
            print(f"[{folder.name}] iet{iet}.ieta{ieta} skipped: {exc}")
        finally:
            gc.collect()

[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 SP/PD/PF -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/model_metrics/iet0.ieta4_performance_sp_pd_pf.pdf


### 6. Task 4: ROC Curves

In [15]:
for folder in RESULTS_FOLDERS:
    if not folder.exists():
        print(f"Skipping missing folder: {folder}")
        continue
    agg = ResultAggregator(folder)
    model_tag = extract_model_tag(folder)
    model_out = OUTPUT_DIR / folder.name
    for iet, ieta in resolve_regions(agg, TARGET_REGIONS):
        try:
            best_model, _, _, _ = agg.get_best_model_for_region(iet, ieta, cache=False)
            cb = (best_model.get("history") or {}).get("callbackMetrics", {})
            roc_path = roc_plotter.plot(
                make_context(iet, ieta, model_out),
                best_model_details=cb,
            )
            dest = move_to_standard_name(
                roc_path,
                model_out / "ROC",
                f"et_{iet}_eta_{ieta}_{model_tag}_roc_plot.pdf",
            )
            print(f"[{folder.name}] et_{iet}_eta_{ieta} ROC -> {dest}")
        except Exception as exc:
            print(f"[{folder.name}] iet{iet}.ieta{ieta} skipped: {exc}")
        finally:
            gc.collect()

[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 ROC -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/ROC/iet0.ieta4_model_ROC.pdf


### 7. Task 5: Cross-Validation Fold Boxplots (`SP`, `PD`, `PF`)

In [16]:
if not INCLUDE_BOXPLOT:
    print("Skipping CV fold boxplots (INCLUDE_BOXPLOT=False).")
else:
    for folder in RESULTS_FOLDERS:
        if not folder.exists():
            print(f"Skipping missing folder: {folder}")
            continue
        agg = ResultAggregator(folder)
        model_tag = extract_model_tag(folder)
        model_out = OUTPUT_DIR / folder.name
        for iet, ieta in resolve_regions(agg, TARGET_REGIONS):
            try:
                region_df = agg._get_region_dataframe(iet, ieta, cache=False)
                ctx = make_context(iet, ieta, model_out)
                for metric_col in BOXPLOT_METRICS:
                    bp_path = boxplot_plotter.plot(ctx, region_df, data_type=metric_col)
                    metric_short = metric_col.replace("best_", "").replace("_value", "")
                    dest = move_to_standard_name(
                        bp_path,
                        model_out / "Boxplots",
                        f"et_{iet}_eta_{ieta}_{model_tag}_boxplot_{metric_short}_plot.pdf",
                    )
                    print(f"[{folder.name}] et_{iet}_eta_{ieta} {metric_col} -> {dest}")
                del region_df
            except Exception as exc:
                print(f"[{folder.name}] iet{iet}.ieta{ieta} skipped: {exc}")
            finally:
                gc.collect()

[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 best_sp_value -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/boxplot_SP/et0.eta4.boxplot_SP.pdf
[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 best_pd_value -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/boxplot_SP/et0.eta4.boxplot_SP.pdf
[config_nameModelV9_HighBatch_100Rings_id20261105222012] iet0.ieta4 best_fa_value -> /eos/home-i04/p/pmourafr/egamma_master/Plots/separate_model_metrics/config_nameModelV9_HighBatch_100Rings_id20261105222012/boxplot_SP/et0.eta4.boxplot_SP.pdf
